# 3.3 波动率类指标

## 学习目标
- 理解布林带（Bollinger Bands）的构造与信号
- 掌握 ATR（真实波动幅度均值）用于止损设置
- 学会用 VIX 和隐含波动率理解市场情绪

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

data = yf.download('AAPL', start='2022-01-01', end='2024-01-01', progress=False)
close = data['Close'].squeeze()
high = data['High'].squeeze()
low = data['Low'].squeeze()

## 1. 布林带 (Bollinger Bands)

$$Middle = SMA_{20}$$
$$Upper = SMA_{20} + 2\sigma_{20}$$
$$Lower = SMA_{20} - 2\sigma_{20}$$

- 价格触及上轨 → 可能超买
- 价格触及下轨 → 可能超卖
- 带宽收窄 → 低波动，可能酝酿突破

In [ ]:
window = 20
n_std = 2

mid = close.rolling(window).mean()
std = close.rolling(window).std()
upper = mid + n_std * std
lower = mid - n_std * std
bandwidth = (upper - lower) / mid  # 带宽（相对）

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(close.index, close.values, label='收盘价', linewidth=1.2, color='steelblue')
ax1.plot(mid.index, mid.values, label='中轨 SMA20', linewidth=1.5, color='orange')
ax1.plot(upper.index, upper.values, label='上轨', linewidth=1, color='gray', linestyle='--')
ax1.plot(lower.index, lower.values, label='下轨', linewidth=1, color='gray', linestyle='--')
ax1.fill_between(close.index, upper.values, lower.values, alpha=0.07, color='gray')
ax1.set_title('AAPL 布林带 (20, 2)', fontsize=13)
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(bandwidth.index, bandwidth.values, color='purple', linewidth=1.2)
ax2.fill_between(bandwidth.index, bandwidth.values, alpha=0.3, color='purple')
ax2.set_title('布林带宽', fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. ATR — 真实波动幅度均值

$$TR = \max(H-L,\ |H-C_{prev}|,\ |L-C_{prev}|)$$
$$ATR = EMA_{14}(TR)$$

ATR 是**止损位设置**的核心工具：
- 止损 = 入场价 - 1.5 × ATR（做多）

In [ ]:
def compute_atr(high, low, close, period=14):
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    atr = tr.ewm(com=period - 1, min_periods=period).mean()
    return atr

atr = compute_atr(high, low, close)
atr_pct = atr / close  # ATR 相对于价格的百分比

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# 示例：某笔买入后的止损线
ax1.plot(close.index, close.values, linewidth=1.2, color='steelblue', label='收盘价')
stop_loss = close - 1.5 * atr
ax1.plot(stop_loss.index, stop_loss.values, linewidth=1, color='red',
          linestyle='--', label='1.5x ATR 止损线')
ax1.set_title('ATR 止损线示例', fontsize=13)
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(atr_pct.index, atr_pct.values * 100, color='darkorange', linewidth=1.2)
ax2.set_title('ATR% (ATR / 收盘价 × 100)', fontsize=11)
ax2.set_ylabel('%')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f'当前 ATR: {atr.iloc[-1]:.2f} USD')
print(f'建议止损距离（1.5×ATR）: {1.5 * atr.iloc[-1]:.2f} USD')

## 3. VIX — 市场恐慌指数

In [ ]:
vix = yf.download('^VIX', start='2020-01-01', end='2024-01-01', progress=False)['Close'].squeeze()
spy = yf.download('SPY', start='2020-01-01', end='2024-01-01', progress=False)['Close'].squeeze()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(spy.index, spy.values, color='steelblue', linewidth=1.2)
ax1.set_title('SPY (S&P500 ETF)', fontsize=12)
ax1.set_ylabel('价格')
ax1.grid(alpha=0.3)

ax2.plot(vix.index, vix.values, color='red', linewidth=1.2)
ax2.axhline(20, color='orange', linestyle='--', alpha=0.6, label='VIX=20（警戒线）')
ax2.axhline(30, color='red', linestyle='--', alpha=0.6, label='VIX=30（高恐慌）')
ax2.fill_between(vix.index, vix.values, 20, where=(vix.values > 20),
                  alpha=0.2, color='red')
ax2.set_title('VIX 恐慌指数', fontsize=12)
ax2.set_ylabel('VIX')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('注意：VIX 与市场通常呈负相关（市场跌时 VIX 升）')

## 🎯 练习

1. 布林带带宽收窄到历史低点后，价格通常会如何运动？找2个历史例子验证。
2. 将 ATR 用于仓位管理：假设每笔最大亏损是总资金的 1%，如何用 ATR 计算仓位比例？
3. 实现 ATR 追踪止損（Trailing Stop）：止损线随价格上涨而上移。

---
**下一模块** → `../04_backtesting/01_simple_backtest.ipynb`